# Customer Churn Prediction
End-to-end walkthrough: cleaning, EDA, model comparison, evaluation and drivers on the **Telco Customer Churn** dataset.

> Run all cells top to bottom (Kernel → Restart & Run All). Put `telco_churn.csv` in `../data/` first (see `data/README.md`).

In [ ]:
import sys
sys.path.append("..")            # so the notebook can import the churn package

import matplotlib.pyplot as plt
import pandas as pd

from churn.config import TARGET
from churn.data import load_raw, clean

DATA_PATH = "../data/telco_churn.csv"
pd.set_option("display.max_columns", 30)

## 1. Load and inspect

In [ ]:
raw = load_raw(DATA_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.dtypes

## 2. Cleaning
`TotalCharges` is stored as text and is blank for customers with `tenure == 0` (new customers not yet billed). We convert it to numeric and set those to 0. `SeniorCitizen` is mapped to Yes/No and the target to 1/0.

In [ ]:
blank = (raw["TotalCharges"].astype(str).str.strip() == "").sum()
print("Blank TotalCharges rows in raw data:", blank)
df = clean(raw)
print("Missing values after cleaning:", int(df.isna().sum().sum()))
print(f"Churn rate: {df[TARGET].mean():.1%}  ({int(df[TARGET].sum())} of {len(df)} customers)")
df.head()

## 3. Exploratory analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    rate = (df.groupby(col)[TARGET].mean() * 100).sort_values(ascending=False)
    ax.bar(range(len(rate)), rate.values, color="#0995C2")
    ax.set_xticks(range(len(rate)), rate.index, rotation=20, ha="right")
    ax.set_ylabel("Churn rate (%)"); ax.set_title(f"Churn rate by {col}")
plt.tight_layout(); plt.show()

In [ ]:
bucket = pd.cut(df["tenure"], [-1, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"])
tenure_rate = (df.groupby(bucket, observed=True)[TARGET].mean() * 100).round(1)
print("Churn rate (%) by tenure bucket (months):")
print(tenure_rate)
tenure_rate.plot(kind="bar", color="#0995C2", ylabel="Churn rate (%)", title="Churn rate by tenure", rot=0)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for label, g in df.groupby(TARGET):
    ax.hist(g["MonthlyCharges"], bins=30, alpha=0.6, label="Churned" if label else "Stayed")
ax.set_xlabel("Monthly charges"); ax.set_title("Monthly charges by churn"); ax.legend(); plt.show()

## 4. Train and compare models
Three models (Logistic Regression, Random Forest, Gradient Boosting) are tuned with 5-fold CV on the **training split only**. Class imbalance is handled with class weights. The decision threshold is tuned on out-of-fold predictions, and the test set is only touched for the final evaluation. This takes a minute or two.

In [ ]:
from churn.train import run_training

results = run_training(DATA_PATH, out_dir="..")

## 5. Results

In [ ]:
comparison = pd.read_csv("../reports/model_comparison.csv")
comparison

In [ ]:
best = results["best_model"]
final = results["test_metrics_tuned_threshold"]
cap = results["top_20pct_capture"]
print(f"Selected model (best CV ROC-AUC): {best}")
print(f"Test ROC-AUC {final['roc_auc']:.3f} | PR-AUC {final['pr_auc']:.3f} | precision {final['precision']:.3f} | "
      f"recall {final['recall']:.3f} | F1 {final['f1']:.3f} | threshold {final['threshold']}")
print(f"Top 20% highest-risk customers capture {cap['captured_churners']:.1%} of churners ({cap['lift']:.1f}x lift)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ["roc_curves", "confusion_matrix", "feature_importance"]):
    ax.imshow(plt.imread(f"../reports/figures/{name}.png")); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Business interpretation
- Use the **top-risk list** (highest predicted probabilities) to prioritise retention outreach: contacting the top 20% reaches far more churners than random targeting (see lift above).
- The most important drivers (see feature importance) point to where retention offers should focus, for example customers on month-to-month contracts or with short tenure.
- The threshold can be moved: lower it to catch more churners (higher recall, more false alarms) or raise it to reduce wasted offers (higher precision). `python -m churn.train --beta 2` favours recall.

**Limitations:** one dataset from a single company, no time dimension, and correlation is not causation, so a retention experiment (A/B test) would be needed to prove an offer reduces churn.